# Student 3 — Defect Detection
## Notebook 01: YOLOv10 (single-stage, NMS-free)

**Algorithm.** YOLOv10 is the single-stage detector described in Section 2.4.1 of the assignment
documentation. Its distinguishing feature is *consistent dual label assignment*: a one-to-many head
supplies dense supervision during training, while a one-to-one head produces a single prediction
per object at inference. This removes Non-Maximum Suppression (NMS) from the inference path and
gives the architecture its low, predictable latency — the property that makes it the candidate for
in-line conveyor inspection.

**Prerequisite.** Run `00_data_preparation.ipynb` first. This notebook consumes
`dataset/yolo/data.yaml` and writes `results/yolov10.json`, which `04_model_comparison.ipynb`
reads.

### 1. Environment

Ultralytics provides the YOLOv10 implementation and pulls in PyTorch. Run the install cell once;
it is safe to re-run. If a CUDA GPU is present it is used automatically — training on CPU works but
is roughly 20–40x slower, so reduce `EPOCHS` in Section 3 if no GPU is detected.

In [1]:
# Run once per environment (remove the leading '#' the first time)
# %pip install -q ultralytics torch torchvision

In [2]:
import json, shutil, time, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from ultralytics import YOLO
import ultralytics

warnings.filterwarnings("ignore")

CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
print("ultralytics :", ultralytics.__version__)
print("torch       :", torch.__version__)
print("device      :", torch.cuda.get_device_name(0) if CUDA else "CPU (training will be slow)")

Creating new Ultralytics Settings v0.0.7 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\User\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics : 8.4.120
torch       : 2.13.0+cpu
device      : CPU (training will be slow)


### 2. Load the prepared dataset

`dataset_info.json` was written by notebook 00. Reading it here guarantees that YOLOv10, Faster
R-CNN and RT-DETR all use the same class order and the same train/val/test split.

In [3]:
def find_work_dir(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "dataset" / "dataset_info.json").is_file():
            return c
        if (c / "Student3-Defect Detection" / "dataset" / "dataset_info.json").is_file():
            return c / "Student3-Defect Detection"
    raise FileNotFoundError("dataset_info.json not found - run 00_data_preparation.ipynb first.")

WORK_DIR    = find_work_dir(Path.cwd())
INFO        = json.loads((WORK_DIR / "dataset" / "dataset_info.json").read_text())
CLASS_NAMES = INFO["class_names"]
DATA_YAML   = Path(INFO["paths"]["yolo_yaml"])
RESULTS_DIR = WORK_DIR / "results"; RESULTS_DIR.mkdir(exist_ok=True)
RUNS_DIR    = WORK_DIR / "runs";    RUNS_DIR.mkdir(exist_ok=True)

assert DATA_YAML.is_file(), f"Missing {DATA_YAML} - re-run notebook 00."
print("Work dir :", WORK_DIR)
print("data.yaml:", DATA_YAML)
print("Split    :", INFO["counts"], "| boxes:", INFO["boxes"])
print("Classes  :", CLASS_NAMES)

Work dir : C:\Users\User\Image-Processing\Student3-Defect Detection
data.yaml: C:\Users\User\Image-Processing\Student3-Defect Detection\dataset\yolo\data.yaml
Split    : {'train': 483, 'val': 138, 'test': 72} | boxes: {'train': 2051, 'val': 598, 'test': 304}
Classes  : ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']


### 3. Training configuration

`yolov10n` (nano) is the default because it trains quickly and is the variant that matches the
real-time, edge-deployment argument made in the documentation. Switch `MODEL_NAME` to
`yolov10s`/`yolov10m` for higher accuracy at the cost of speed. `imgsz` is 512 to match the
preprocessed images exactly — no further resizing distortion is introduced.

In [4]:
MODEL_NAME = "yolov10n"     # yolov10n | yolov10s | yolov10m | yolov10b | yolov10l | yolov10x
EPOCHS     = 100 if CUDA else 20
IMGSZ      = INFO["image_size"]     # 512
BATCH      = 16 if CUDA else 4
PATIENCE   = 30                     # early stopping
SEED       = INFO["random_seed"]
RUN_NAME   = f"{MODEL_NAME}_pcb"

print(f"{MODEL_NAME} | epochs={EPOCHS} | imgsz={IMGSZ} | batch={BATCH} | device={DEVICE}")

yolov10n | epochs=20 | imgsz=512 | batch=4 | device=cpu


### 4. Load the pretrained model

COCO-pretrained weights are downloaded on first use and fine-tuned on the PCB data — transfer
learning is essential here because 693 images are far too few to train a detector from scratch.
If the download is blocked by a firewall, the fallback builds the same architecture from its YAML
definition and trains from random initialisation (expect noticeably lower mAP).

In [5]:
try:
    model = YOLO(f"{MODEL_NAME}.pt")
    pretrained = True
except Exception as exc:
    print("Could not fetch pretrained weights:", exc)
    model = YOLO(f"{MODEL_NAME}.yaml")
    pretrained = False

n_params = sum(p.numel() for p in model.model.parameters())
print(f"Pretrained: {pretrained} | parameters: {n_params/1e6:.2f} M")

Pretrained: True | parameters: 2.78 M


### 5. Train

Ultralytics handles the mosaic/HSV augmentation, the dual-assignment loss and the learning-rate
schedule internally. Every artefact of the run (weights, curves, confusion matrix, sample batches)
is written under `runs/<RUN_NAME>/`. The wall-clock training time is recorded for the comparison
table in notebook 04.

In [6]:
t0 = time.perf_counter()
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    seed=SEED,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=pretrained,
    val=True,
    plots=True,
    verbose=True,
)
train_time = time.perf_counter() - t0

RUN_DIR   = Path(train_results.save_dir)
BEST_PT   = RUN_DIR / "weights" / "best.pt"
print(f"\nTraining finished in {train_time/60:.1f} min")
print("Best weights:", BEST_PT)

Ultralytics 8.4.120  Python-3.12.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13500HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\User\Image-Processing\Student3-Defect Detection\dataset\yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.

### 6. Training curves

The loss curves and the validation mAP curve show whether the model converged and whether it began
to overfit — with fewer than 700 images, overfitting is the main risk. A validation mAP that
plateaus while the training loss keeps falling is the signature to look for.

In [7]:
import csv

csv_path = RUN_DIR / "results.csv"
rows = list(csv.DictReader(csv_path.open()))
hist = {k.strip(): [float(r[k]) for r in rows] for k in rows[0]}

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for key in [k for k in hist if k.startswith("train/") and "loss" in k]:
    ax[0].plot(hist["epoch"], hist[key], label=key.split("/")[-1])
ax[0].set_title("Training losses"); ax[0].set_xlabel("epoch"); ax[0].legend()

for key in [k for k in hist if k.startswith("val/") and "loss" in k]:
    ax[1].plot(hist["epoch"], hist[key], label=key.split("/")[-1])
ax[1].set_title("Validation losses"); ax[1].set_xlabel("epoch"); ax[1].legend()

for key, lab in [("metrics/mAP50(B)", "mAP@0.5"), ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    if key in hist:
        ax[2].plot(hist["epoch"], hist[key], label=lab)
ax[2].set_title("Validation mAP"); ax[2].set_xlabel("epoch"); ax[2].legend()
plt.tight_layout(); plt.show()

<Figure size 1600x400 with 3 Axes>

### 7. Evaluation on the held-out test split

The test split was never seen during training or early stopping, so these are the numbers reported
in Chapter 4 of the documentation. `mAP@0.5` is the headline detection accuracy; `mAP@0.5:0.95`
additionally rewards tight localisation, which matters for the very small PCB defects.

In [8]:
best = YOLO(str(BEST_PT))
metrics = best.val(
    data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, project=str(RUNS_DIR), name=f"{RUN_NAME}_test",
    exist_ok=True, plots=True, verbose=False,
)

box = metrics.box
summary = {
    "precision":  float(box.mp),
    "recall":     float(box.mr),
    "mAP50":      float(box.map50),
    "mAP75":      float(box.map75),
    "mAP50_95":   float(box.map),
}
summary["f1"] = (2 * summary["precision"] * summary["recall"] /
                 max(summary["precision"] + summary["recall"], 1e-9))

print(f"{'metric':<14}{'value':>9}")
for k, v in summary.items():
    print(f"{k:<14}{v:>9.4f}")

Ultralytics 8.4.120  Python-3.12.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13500HX)
YOLOv10n summary (fused): 102 layers, 2,266,338 parameters, 0 gradients, 6.6 GFLOPs
WARNING val: Slow image access detected (ping: 0.20.0 ms, read: 11.70.9 MB/s, size: 113.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Users\User\Image-Processing\Student3-Defect Detection\dataset\yolo\labels\test... 72 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 420.6it/s 0.2s0.2s
val: New cache created: C:\Users\User\Image-Processing\Student3-Defect Detection\dataset\yolo\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 5.3it/s 3.4s0.2s
                   all         72        304      0.565      0.527      0.566      0.249
Speed: 0.6ms preprocess, 40.6ms inference, 0.0ms loss, 0.1ms postprocess per image
R

### 8. Per-class results

Per-class AP exposes which defect types the detector struggles with. `open_circuit` and
`mouse_bite` are typically the hardest because they are thin, low-contrast breaks in a copper
track, whereas `missing_hole` is a high-contrast circular difference.

In [9]:
per_class = {}
for idx, cls_idx in enumerate(box.ap_class_index):
    name = CLASS_NAMES[int(cls_idx)]
    per_class[name] = {
        "precision": float(box.p[idx]),
        "recall":    float(box.r[idx]),
        "mAP50":     float(box.ap50[idx]),
        "mAP50_95":  float(box.ap[idx].mean()) if box.ap[idx].ndim else float(box.ap[idx]),
    }

print(f"{'class':<18}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>10}")
for n, m in per_class.items():
    print(f"{n:<18}{m['precision']:>8.3f}{m['recall']:>8.3f}{m['mAP50']:>9.3f}{m['mAP50_95']:>10.3f}")

names = list(per_class)
x = np.arange(len(names))
plt.figure(figsize=(9, 4))
plt.bar(x - 0.2, [per_class[n]["mAP50"] for n in names], 0.4, label="mAP@0.5")
plt.bar(x + 0.2, [per_class[n]["mAP50_95"] for n in names], 0.4, label="mAP@0.5:0.95")
plt.xticks(x, names, rotation=30, ha="right"); plt.ylim(0, 1)
plt.title(f"{MODEL_NAME} - per-class AP on the test split"); plt.legend(); plt.tight_layout(); plt.show()

class                    P       R    mAP50  mAP50-95
missing_hole         0.651   0.835    0.821     0.449
mouse_bite           0.610   0.407    0.602     0.228
open_circuit         0.405   0.387    0.325     0.163
short                0.536   0.627    0.627     0.234
spur                 0.649   0.462    0.563     0.216
spurious_copper      0.540   0.446    0.460     0.205


<Figure size 900x400 with 1 Axes>

### 9. Inference speed

Objective 1 and Objective 3 of the documentation are stated in frames per second, so latency is
measured explicitly rather than taken from the training log. Ten warm-up images are discarded
(first-call CUDA/cuDNN initialisation) before timing the remaining test images one at a time, which
is the honest single-image latency an in-line inspection station would see.

In [10]:
test_images = sorted((Path(INFO["paths"]["yolo_root"]) / "images" / "test").glob("*.jpg"))
warmup, timed = test_images[:10], test_images

for p in warmup:
    best.predict(str(p), imgsz=IMGSZ, device=DEVICE, verbose=False)

latencies = []
for p in timed:
    t = time.perf_counter()
    best.predict(str(p), imgsz=IMGSZ, device=DEVICE, verbose=False)
    latencies.append((time.perf_counter() - t) * 1000)

lat_mean, lat_std = float(np.mean(latencies)), float(np.std(latencies))
fps = 1000.0 / lat_mean
print(f"Images timed        : {len(timed)}")
print(f"Latency per image   : {lat_mean:.2f} +/- {lat_std:.2f} ms")
print(f"Throughput          : {fps:.1f} FPS  (NMS-free end-to-end)")
print(f"Ultralytics profile : {metrics.speed}")

Images timed        : 72
Latency per image   : 52.13 +/- 6.68 ms
Throughput          : 19.2 FPS  (NMS-free end-to-end)
Ultralytics profile : {'preprocess': 0.5705777777317659, 'inference': 40.55557361114855, 'loss': 0.00030277783480414655, 'postprocess': 0.10484305554806876}


### 10. Qualitative results

Six test images are shown with the predicted boxes (confidence >= 0.25). Visual inspection catches
failure modes that a single mAP number hides — for example duplicate boxes, or defects found on the
silkscreen instead of the copper.

In [11]:
sample = test_images[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, p in zip(axes.ravel(), sample):
    r = best.predict(str(p), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)[0]
    ax.imshow(Image.open(p))
    for b, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                        r.boxes.cls.cpu().numpy().astype(int),
                        r.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = b
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                   edgecolor="red", linewidth=1.8))
        ax.text(x1, max(y1 - 4, 8), f"{CLASS_NAMES[c]} {cf:.2f}", color="yellow", fontsize=7,
                bbox=dict(facecolor="black", alpha=.6, pad=1))
    ax.set_title(p.stem, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

<Figure size 1600x1100 with 6 Axes>

### 11. Confusion matrix

Ultralytics saves a normalised confusion matrix for the test run. The background row/column is the
important one: entries there are **missed defects** (false negatives) and **false alarms**, the two
error types with real cost on a production line.

In [12]:
cm_path = RUN_DIR.parent / f"{RUN_NAME}_test" / "confusion_matrix_normalized.png"
if cm_path.exists():
    plt.figure(figsize=(8, 7)); plt.imshow(Image.open(cm_path)); plt.axis("off"); plt.show()
else:
    print("Confusion matrix image not found at", cm_path)

<Figure size 800x700 with 1 Axes>

### 12. Save the results

Everything notebook 04 needs is written to `results/yolov10.json` in a schema shared by all three
model notebooks, so the comparison table can be assembled without re-running any training.

In [13]:
payload = {
    "model": "YOLOv10",
    "variant": MODEL_NAME,
    "framework": f"ultralytics {ultralytics.__version__}",
    "type": "single-stage (NMS-free)",
    "pretrained": pretrained,
    "params_M": round(n_params / 1e6, 2),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "device": "GPU" if CUDA else "CPU",
    "train_time_min": round(train_time / 60, 2),
    "test": summary,
    "per_class": per_class,
    "latency_ms": round(lat_mean, 2),
    "latency_std_ms": round(lat_std, 2),
    "fps": round(fps, 1),
    "weights": str(BEST_PT),
}
out = RESULTS_DIR / "yolov10.json"
out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved", out)
print(json.dumps({k: v for k, v in payload.items() if k != "per_class"}, indent=2))

Saved C:\Users\User\Image-Processing\Student3-Defect Detection\results\yolov10.json
{
  "model": "YOLOv10",
  "variant": "yolov10n",
  "framework": "ultralytics 8.4.120",
  "type": "single-stage (NMS-free)",
  "pretrained": true,
  "params_M": 2.78,
  "epochs": 20,
  "imgsz": 512,
  "batch": 4,
  "device": "CPU",
  "train_time_min": 37.37,
  "test": {
    "precision": 0.5652321541022092,
    "recall": 0.5274152449268242,
    "mAP50": 0.5664828576416527,
    "mAP75": 0.14742887543473635,
    "mAP50_95": 0.24914399809814664,
    "f1": 0.5456692712786329
  },
  "latency_ms": 52.13,
  "latency_std_ms": 6.68,
  "fps": 19.2,
  "weights": "C:\\Users\\User\\Image-Processing\\Student3-Defect Detection\\runs\\yolov10n_pcb\\weights\\best.pt"
}
